# Paper 5, RB-5 - key-block test: is the top rebuilder a fixed head, or just the first block after the window?

**Naming.** W1-W7 = September runs. RB = November rebuttal runs: RB-0 single heads (`w8_*`, canonical = cuDNN 9.10.2
re-run), RB-1/1b drift checks, RB-2 axis map, RB-3 intact/shuffled/null, RB-4 mixed RoPE. This is **RB-5**; it writes
to `./rb5_output/`.

**Why.** Every rebuilding result in the paper uses one window, rotations off in blocks 0-5, and RB-2 to RB-4 found the
top rebuilder in **block 6, the first block after that window**, on most models: naver (6,6), reg1-gap (6,9), DINOv3
(6,2), mixed RoPE (6,5); ViT-S (7,4)/(7,2) next to it. That leaves one reading we have not excluded: the "key" head may
be key only because it sits at the window's edge, so that any window would make *its* first block look special.
W3's window sweep (0-2 / 0-5 / 0-8) was measured at whole-model level, so it cannot tell these apart.

RB-5 moves the window edge and measures every single head again, with the same rules as RB-0 and RB-2:

| window | first block after it | single heads measured | references |
|---|---|---|---|
| blocks 0-3 | 4 | every (block, head) in 4-11 | W3 `prefix_3` (naver, DINOv3) |
| blocks 0-5 | 6 | block 6 re-measured as a gate; the rest is RB-0 | RB-0 A / C (= W5) |
| blocks 0-7 | 8 | every (block, head) in 8-11 | W3 `prefix_7` (naver, DINOv3) |

For each: SSDC (repack, 1,000 images) at every readout, and top-1 (ILSVRC, 5,000 images) on the three classifiers.
Heads are ranked by RB-2's pre-registered rule (SSDC lost entering block 11; block-11 heads separately at the output
of block 11), and the reading is fixed below before the run.

**Coherence.** Same env (`env_paper5_cudnn910`), datasets, counts, batch, seed, fp32 and code paths as RB-0/RB-3. Gates
demand bit-identical agreement with W3 for the two new windows (naver, DINOv3), and with RB-0 for the 0-5 window and
the block-6 heads. Runtime ~45-60 min on one H100.

In [ ]:
import importlib, subprocess, sys
def ensure(p, m=None):
    try:
        importlib.import_module(m or p); return False
    except ImportError:
        print(f"installing {p} ..."); subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p]); return True
for p in ["timm", "datasets", "scipy", "pandas"]:
    ensure(p)
print("environment ready")

In [ ]:
import os, json, time, warnings, traceback, shutil, hashlib, platform, socket
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
warnings.filterwarnings("ignore", category=UserWarning)
print("python", sys.executable)
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info(); print(f"memory: {free/1e9:.1f} / {total/1e9:.1f} GB free")

In [ ]:
# config
REPO_PATH  = os.environ.get("RB5_REPO", "")
RB0_DIR    = os.environ.get("RB5_RB0_DIR", "")
MODELS_RUN = [m for m in os.environ.get("RB5_MODELS", "naver,DINOv3,small,reg1_gap").split(",") if m]
RESUME     = os.environ.get("RB5_RESUME", "1") == "1"
STRICT     = os.environ.get("RB5_STRICT", "1") == "1"
TEST_MODE  = os.environ.get("RB5_TEST", "0") == "1"   # CPU smoke test; needs a test RB-0 made by a seeded W8 copy
OUT_DIR    = Path(os.environ.get("RB5_OUT", "rb5_output"))

N_SSDC, N_ACC, BATCH_SIZE = 1000, 5000, 256
METRIC, TORCH_SEED, INPUT_SIZE = "manhattan", 20260908, 224
SSDC_DATASET = "benjamin-paine/imagenet-1k-256x256"
ACC_DATASET  = "ILSVRC/imagenet-1k"
N_BLOCKS = 12
WINDOW   = list(range(0, 6))       # the paper's window (RB-0)
POST     = list(range(6, 12))
FINAL    = 12
PROBE_LR, PROBE_PASSES, PROBE_BATCH, PROBE_TRAIN_FRAC = 5e-3, 20, 512, 0.8   # unused here; measure()'s cell defines them
PROBE_CHUNK, PROBE_DEVICE = 8192, None
NOISE_SEEDS, NOISE_BOOT = [0, 1, 2], 200

WINDOWS = {"w3": list(range(0, 4)), "w5": list(range(0, 6)), "w7": list(range(0, 8))}   # name = last masked block
W3_NAME = {"w3": "prefix_3", "w5": "prefix_5", "w7": "prefix_7"}                        # W3's own names (blocks 0..k)
W3_FILE = "w3/w3_windows_2026-09-12T09-59-03Z.json"
W3_KEY  = {"naver": "RoPE · naver", "DINOv3": "RoPE · DINOv3"}

EXACT_TOL = 0.0
EXPECT_TORCH, EXPECT_CUDNN = "2.10.0+cu128", 91002
EXPECT_CUDNN_PKGS = {"nvidia-cudnn-cu12": "9.10.2.21"}
FIRST_LAYER_910 = {"naver": 0.00041303808169361526, "DINOv3": 0.0004745740108653513,
                   "small": 0.000510305810057617, "reg1_gap": 0.00026782853342761795}
RB0_SHA256 = {
    "w8_partA_naver.json":     "f50b694d24279ce63b8dac8da1906b7e60ab6522ce8de4c992b1c87ceed10914",
    "w8_partA_DINOv3.json":    "f1d963652ac3867c79bb170da2b2dd86ff8e978008f5ff59304a69947bb909b5",
    "w8_partA_small.json":     "be01ed6297ef819a8fc219853c07fc9a81a76d2c6ffbe9da8501379fdf6acbbb",
    "w8_partA_reg1_gap.json":  "da5ae49a2abc21aa0df8f4c43b249727a7edfe444254e239e637ed96a312fefb",
    "w8_partC_naver.json":     "c983414404ca645bc8144a2bd4ec76bfb583f4c5c9588ec8faffe853e5c3b985",
    "w8_partC_small.json":     "bf895274486dfaa9ab042118e15bb7c9863a5f6201ab27ec11fb539ecf7126f6",
    "w8_partC_reg1_gap.json":  "8b5641235e05b20bfdc597032097b2f85878c228bdc32ec8b63888227e393698",
    "w8_perimage_naver.npz":   "5fe4a53ecbf99bc583c56ef6ec9f774c5bd6027e1734dce16fe0c230c36358c6",
    "w8_perimage_small.npz":   "b9f6a3833766b73618c82b3e5ecbb81af672b11eaa49fab503e27bb8734cf40b",
    "w8_perimage_reg1_gap.npz": "10310f0c0109d65f0b594f7774006ddd2c69f5fc74438b13d32956d39738643e",
}
W3_SHA256 = "a83dc9d85ce0c99c8b035058fc6ab1614b5049ac336d8335b3bb72f7256c6bb5"
MODEL_SPECS = {
    "naver":    dict(name="vit_base_patch16_rope_224.naver_in1k",        classifier=True,  label="naver ViT-B"),
    "DINOv3":   dict(name="vit_base_patch16_dinov3.lvd1689m",            classifier=False, label="DINOv3 ViT-B"),
    "small":    dict(name="vit_small_patch16_rope_224.naver_in1k",       classifier=True,  label="naver ViT-S"),
    "reg1_gap": dict(name="vit_base_patch16_rope_reg1_gap_256.sbb_in1k", classifier=True,  label="reg1-gap ViT-B"),
}
RB0_FILES = {k: [f"w8_partA_{k}.json"] + ([f"w8_partC_{k}.json", f"w8_perimage_{k}.npz"] if s["classifier"] else [])
             for k, s in MODEL_SPECS.items()}
EXPECT_TOP = {"naver": [(6, 6), (7, 2), (6, 1), (6, 2), (8, 10)], "DINOv3": [(6, 2), (9, 10), (7, 9), (10, 0), (10, 8)],
              "small": [(7, 2), (7, 4), (6, 0), (7, 3), (8, 4)], "reg1_gap": [(6, 9), (7, 7), (6, 7), (6, 4), (7, 6)]}
TOP_SHUFFLE = 4            # used by the shared selection cell only
REPLAY_CHECK_N = 512

# rules and reading, fixed before the run
RULE_RANK = ("within each window, single heads in the blocks after it up to block 10 are ranked by SSDC lost entering block 11 "
             "vs that window's reference; block-11 heads separately, at the output of block 11 (RB-2's pre-registered rule)")
RULE_READING = (
    "Window 0-3 is the test (the 0-5 top heads are outside it, so they can be compared directly). "
    "FIXED HEAD if, under 0-3, the 0-5 top head (by the rule) is still in the top 3 by SSDC AND, on the classifiers, in the top 3 by "
    "top-1 cost. EDGE-FOLLOWING if, under 0-3, the top head by SSDC is in block 4 and the 0-5 top head is outside the top 10 by "
    "SSDC. Anything else is reported as MIXED, with the ranks. Window 0-7 cannot test the 0-5 top head (it lies inside the "
    "window); it shows whether block 8 takes over when it is the first block after the window.")
PERM_N = 10000

if TEST_MODE:
    N_SSDC, N_ACC, BATCH_SIZE = 24, 32, 8
    PERM_N = 500
    OUT_DIR = Path(os.environ.get("RB5_OUT", "rb5_test_output"))
OUT_DIR.mkdir(exist_ok=True, parents=True)
_bad = [m for m in MODELS_RUN if m not in MODEL_SPECS]
if _bad: raise SystemExit(f"Unknown entries in RB5_MODELS: {_bad}")

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H-%M-%SZ")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PDEV = PROBE_DEVICE or DEVICE
GATES = []
GATE_FILE = OUT_DIR / f"rb5_gates_session_{RUN_STAMP}.json"
def gate(name, scope, passed, detail="", skip=False):
    status = "SKIP" if skip else ("PASS" if passed else "FAIL")
    GATES.append({"gate": name, "scope": scope, "status": status, "detail": detail, "session": RUN_STAMP})
    print(f"  [{status}] {scope}: {name}" + (f": {detail}" if detail else ""), flush=True)
    GATE_FILE.write_text(json.dumps(GATES, indent=1))
    return bool(passed) or skip
print(f"device {DEVICE} | fp32 | models {MODELS_RUN} | windows {WINDOWS}")
print(f"SSDC: {SSDC_DATASET} x {N_SSDC} | top-1: {ACC_DATASET} x {N_ACC} | batch {BATCH_SIZE} | seed {TORCH_SEED}")
print(f"output {OUT_DIR.resolve()} | resume {RESUME} | strict {STRICT} | test mode {TEST_MODE} | run {RUN_STAMP}")

## Pre-flight: are we in `env_paper5_cudnn910`?
A small GPU convolution loads cuDNN; then the notebook checks the version PyTorch reports, which library files were
actually loaded (from the process's own memory map), the installed cuDNN packages, and that the TF32 / cuDNN flags
are PyTorch's defaults (W5 and RB-0 ran with defaults). Nothing is measured if any check fails.

In [ ]:
def _sh(cmd, timeout=120):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return ((r.stdout or "") + (r.stderr or "")).strip()
    except Exception as ex:
        return f"(failed: {ex!r})"

def _loaded_cudnn():
    try:
        with open("/proc/self/maps") as f:
            return sorted({l.split()[-1] for l in f if "libcudnn" in l and l.split()[-1].startswith("/")})
    except Exception:
        return []

import importlib.metadata as _md
if DEVICE == "cuda":
    _x = torch.randn(2, 3, 32, 32, device="cuda"); _w = torch.randn(8, 3, 16, 16, device="cuda")
    F.conv2d(_x, _w, stride=16); torch.cuda.synchronize(); del _x, _w
CUDNN_PKGS = {}
for _d in _md.distributions():
    _n = (_d.metadata.get("Name") or "").lower()
    if "cudnn" in _n: CUDNN_PKGS[_n] = _d.version
ENV = {
    "python": sys.executable, "prefix": sys.prefix, "host": socket.gethostname(), "platform": platform.platform(),
    "torch": torch.__version__, "timm": timm.__version__, "numpy": np.__version__,
    "cudnn_reported": torch.backends.cudnn.version() if DEVICE == "cuda" else None,
    "libcudnn_loaded": _loaded_cudnn(), "cudnn_packages": CUDNN_PKGS,
    "flags": {"matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32, "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
              "cudnn_benchmark": torch.backends.cudnn.benchmark, "cudnn_deterministic": torch.backends.cudnn.deterministic,
              "float32_matmul_precision": torch.get_float32_matmul_precision()},
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "gpu": _sh("nvidia-smi --query-gpu=index,name,uuid,driver_version,memory.used,memory.total --format=csv") if DEVICE == "cuda" else "cpu",
}
(OUT_DIR / f"rb5_pip_freeze_{RUN_STAMP}.txt").write_text(_sh(f"{sys.executable} -m pip freeze", timeout=300))
(OUT_DIR / f"rb5_environment_{RUN_STAMP}.json").write_text(json.dumps(ENV, indent=1, default=str))
print(json.dumps({k: v for k, v in ENV.items() if k != "gpu"}, indent=1, default=str)); print(ENV["gpu"])

_pre = []
if TEST_MODE:
    gate("pre-flight (cuDNN 9.10.2 environment)", "env", True, "test mode on CPU", skip=True)
else:
    _pre.append(gate("CUDA available", "env", DEVICE == "cuda"))
    _pre.append(gate(f"torch is {EXPECT_TORCH}", "env", torch.__version__ == EXPECT_TORCH, torch.__version__))
    _pre.append(gate(f"PyTorch reports cuDNN {EXPECT_CUDNN} (9.10.2)", "env", ENV["cudnn_reported"] == EXPECT_CUDNN,
                     str(ENV["cudnn_reported"])))
    _prefix = os.path.realpath(sys.prefix)
    _libs = ENV["libcudnn_loaded"]
    _pre.append(gate("every loaded libcudnn file comes from this environment's nvidia/cudnn folder", "env",
                     bool(_libs) and all(os.path.realpath(l).startswith(_prefix) and "/nvidia/cudnn/" in l for l in _libs),
                     "; ".join(Path(l).name for l in _libs) + f" (prefix {_prefix})"))
    _pre.append(gate("the only cuDNN package installed is nvidia-cudnn-cu12 9.10.2.21 (no cu13)", "env",
                     CUDNN_PKGS == EXPECT_CUDNN_PKGS, str(CUDNN_PKGS)))
    _f = ENV["flags"]
    _pre.append(gate("TF32 / cuDNN flags are PyTorch's defaults (as in W5 and RB-0)", "env",
                     _f["matmul_allow_tf32"] is False and _f["cudnn_allow_tf32"] is True and _f["cudnn_benchmark"] is False
                     and _f["cudnn_deterministic"] is False and _f["float32_matmul_precision"] == "highest", str(_f)))
PREFLIGHT_OK = all(_pre)
if not PREFLIGHT_OK:
    raise SystemExit("PRE-FLIGHT FAILED: this notebook must run in env_paper5_cudnn910 (cuDNN 9.10.2), with PyTorch's "
                     "default flags. Nothing was measured. Switch the kernel / python and run again.")
print("\npre-flight OK")

In [ ]:
def resolve_repo():
    cands = []
    if REPO_PATH:
        cands.append(Path(REPO_PATH).expanduser())
    here = Path.cwd()
    for base in [here, *list(here.parents)[:4]]:
        cands += [base / "vit-sae-analysis" / "project_code" / "src", base / "project_code" / "src", base / "src", base]
    for c in cands:
        c = c.resolve()
        if (c / "metrics").exists() and (c / "main").exists() and (c / "experiments").exists():
            return c
    raise FileNotFoundError("No checkout with metrics/, main/, experiments/. Set RB5_REPO. Tried: " +
                            ", ".join(str(c) for c in cands[:8]))

repo = resolve_repo()
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
REPO_COMMIT = _sh(f"git -C {repo} rev-parse HEAD")
print("repo root:", repo, "| commit", REPO_COMMIT)

from metrics.ssdc import (evaluate_ssdc, spatial_similarity_distance_correlation, SSDCAccumulator,
                          _register_ssdc_accumulator_hooks)
from main.load_models import get_vit_blocks, get_block_attention, get_patch_embed_conv, load_rope
from main.model import predict
from main.prep_data import prep_data
from experiments.common import load_imagenet

from scipy.spatial.distance import cdist as _cdist
from scipy.stats import rankdata
_G = 14
_c = np.stack(np.meshgrid(np.arange(_G), np.arange(_G), indexing="ij"), -1).reshape(-1, 2)
_D = _cdist(_c, _c, metric="cityblock")
_a = spatial_similarity_distance_correlation(-_D, grid_size=_G, metric="manhattan")
_b = spatial_similarity_distance_correlation(_D, grid_size=_G, metric="manhattan")
_r = spatial_similarity_distance_correlation(np.random.default_rng(0).normal(size=(_G * _G,) * 2),
                                             grid_size=_G, metric="manhattan")
gate("imported metric satisfies its unit test", "repo-metric",
     _a > 0.999 and _b < -0.999 and abs(_r) < 0.1, f"{_a:+.3f}/{_b:+.3f}/{_r:+.3f}")

## The canonical RB-0 (fingerprint check)
RB-5 is gated against RB-0 parts A and C and reuses their numbers, so it must read the **cuDNN 9.10.2 re-run**
(`rb0_cudnn910/w8_output`). Every file it reads is checked by sha256.

In [ ]:
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

RB0 = {}
if not RB0_DIR and not TEST_MODE:
    raise SystemExit("Set RB5_RB0_DIR to the canonical RB-0 folder: "
                     "/workspace/NRDL/experiments/rb0_cudnn910/w8_output")
_rb0 = Path(RB0_DIR).expanduser() if RB0_DIR else None
_rb0_bad = []
for key in MODELS_RUN:
    for fn in RB0_FILES[key]:
        p = (_rb0 / fn) if _rb0 else None
        if p is None or not p.exists():
            if TEST_MODE: gate(f"RB-0 file {fn} present", key, True, "test mode: no RB-0 folder", skip=True)
            else: gate(f"RB-0 file {fn} present", key, False, str(p)); _rb0_bad.append(fn)
            continue
        if TEST_MODE:
            gate(f"{fn} is the canonical RB-0 file", key, True, "test mode: hash not checked", skip=True)
        else:
            d = sha256_file(p)
            if not gate(f"{fn} is the canonical RB-0 file (cuDNN 9.10.2 re-run)", key, d == RB0_SHA256[fn],
                        f"sha256 {d[:16]}... (expected {RB0_SHA256[fn][:16]}...)"):
                _rb0_bad.append(fn)
if _rb0_bad:
    raise SystemExit(f"RB-0 FINGERPRINT FAILED for {_rb0_bad}. RB5_RB0_DIR must be the cuDNN 9.10.2 re-run "
                     "(rb0_cudnn910/w8_output, 4 Nov). Nothing was measured.")
for key in MODELS_RUN:
    d = {}
    for part in "ACD":
        p = (_rb0 / f"w8_part{part}_{key}.json") if _rb0 else None
        d[part] = json.loads(p.read_text()) if (p is not None and p.exists()) else None
    p = (_rb0 / f"w8_perimage_{key}.npz") if _rb0 else None
    d["P"] = ({n: z_[n] for z_ in [np.load(p)] for n in z_.files} if (p is not None and p.exists()) else None)
    RB0[key] = d
    need = "AC" if MODEL_SPECS[key]["classifier"] else "A"
    gate("RB-0 parts needed by RB-5 are present and complete", key,
         all((d[x] or {}).get("complete") for x in need) and (d["P"] is not None or not MODEL_SPECS[key]["classifier"]),
         ", ".join(f"{x}={'complete' if (d[x] or {}).get('complete') else 'missing'}" for x in need))

# the pre-registered selection, recomputed from RB-0 part A
def rank_ssdc_b11(RA):
    c = RA["conditions"]; nh = RA["n_heads"]; cref = c["Cref"]["ssdc"]
    r = sorted([(cref[11] - c[f"s_b{k}_h{h}"]["ssdc"][11], k, h) for k in POST[:-1] for h in range(nh)], key=lambda t: (-t[0], t[1], t[2]))
    r11 = sorted([(cref[FINAL] - c[f"s_b11_h{h}"]["ssdc"][FINAL], 11, h) for h in range(nh)], key=lambda t: (-t[0], t[2]))
    return [(k, h) for _, k, h in r], [(k, h) for _, k, h in r11], {(k, h): v for v, k, h in r + r11}
SEL = {}
for key in MODELS_RUN:
    if not RB0[key]["A"]: continue
    r610, r11, cost = rank_ssdc_b11(RB0[key]["A"])
    best = {k: [max(range(RB0[key]["A"]["n_heads"]), key=lambda h: (cost[(k, h)], -h))] for k in POST[:-1]}
    best[11] = [r11[0][1]]
    top6 = {}
    for k, h in r610[:6]: top6.setdefault(k, []).append(h)
    SEL[key] = {"rank_6_10": r610, "rank_11": r11, "cost": cost, "top": r610[:TOP_SHUFFLE], "best_per_block": best, "top6": top6}
    if TEST_MODE:
        gate("pre-registered top heads equal RB-2's printed ranking", key, True, f"test mode; top {r610[:5]}", skip=True)
    else:
        gate("pre-registered top heads (ssdc_b11, from RB-0 A) equal RB-2's printed ranking", key,
             [tuple(x) for x in r610[:5]] == EXPECT_TOP[key], f"{r610[:5]}")
    print(f"  {key}: top {TOP_SHUFFLE} {r610[:TOP_SHUFFLE]} | best per block {best} | top6 {top6}")

In [ ]:
# committed W5 and W3 anchors
RES_ROOT = repo.parents[1] / "results" / "runs" / "imagenet1k_val" / "rope_interventions"
def _first(pattern):
    hits = sorted(RES_ROOT.glob(pattern)); return hits[0] if hits else None
W5_FILES = {nm: _first(f"w5/w5_partA_{nm}_*.json") for nm in MODEL_SPECS}
ANCH, W3 = {}, {}
if TEST_MODE:
    gate("committed W5 / W3 anchors loaded", "anchors", True, "test mode: tiny random models have no anchors", skip=True)
else:
    f3 = RES_ROOT / W3_FILE
    if not f3.exists():
        gate("committed W3 file found", "W3", False, str(f3))
    else:
        gate("committed W3 file is the one RB-5 was written against (sha256)", "W3", sha256_file(f3) == W3_SHA256, sha256_file(f3)[:16] + "...")
        e3 = json.load(open(f3))
        for nm, k3 in W3_KEY.items():
            if nm in MODELS_RUN:
                c3 = e3["results"][k3]["conditions"]
                W3[nm] = {w: c3[W3_NAME[w]]["ssdc"] for w in WINDOWS}
                W3[nm]["baseline"] = c3["baseline"]["ssdc"]
                gate("W3's own windows are blocks 0-3 / 0-5 / 0-7", nm,
                     all(c3[W3_NAME[w]]["window"] == WINDOWS[w] for w in WINDOWS), str({w: c3[W3_NAME[w]]["window"] for w in WINDOWS}))
for nm in ([] if TEST_MODE else MODELS_RUN):
    f = W5_FILES.get(nm)
    if f is None: gate("committed W5 file found", nm, False, "missing"); continue
    e = json.load(open(f))
    ANCH[nm] = {"file": f.name, "n_heads": e["n_heads"], "row_planes": e["row_planes"], "col_planes": e["col_planes"],
                "stageC": [d["head"] for d in e["stageC_ranking"]], "cref": e["conditions"]["Cref_prefix5"]["ssdc"]}
    if nm in W3:
        gate("W3 prefix_5 and baseline equal W5 bit for bit (same protocol, so W3's other windows are usable anchors)", nm,
             W3[nm]["w5"] == ANCH[nm]["cref"] and W3[nm]["baseline"] == e["conditions"]["baseline"]["ssdc"])
print("W5:", {k: v["file"] for k, v in ANCH.items()}, "| W3 windows for:", list(W3))

In [ ]:
# datasets: explicit ids, no silent fallback
if TEST_MODE:
    from PIL import Image as _PIL
    class _FakeStream:
        def __init__(self, n, seed):
            self.n, self.seed = n, seed
        def __iter__(self):
            rng = np.random.default_rng(self.seed)
            for i in range(self.n):
                arr = rng.integers(0, 256, size=(256, 256, 3), dtype=np.uint8)
                yield {"image": _PIL.fromarray(arr), "label": int(i % 10)}
    DS_SSDC = _FakeStream(400, 1); DS_ACC = _FakeStream(400, 2)
    print("TEST MODE: random images stand in for both datasets")
else:
    DS_SSDC = load_imagenet(split="validation", streaming=True, dataset_id=SSDC_DATASET, fallback_id=None)
    DS_ACC = load_imagenet(split="validation", streaming=True, dataset_id=ACC_DATASET, fallback_id=None)
    gate("SSDC dataset is the repack (W2-W6, RB-0, RB-2)", "data", SSDC_DATASET == "benjamin-paine/imagenet-1k-256x256", SSDC_DATASET)
    gate("top-1 dataset is the ILSVRC split (W7, RB-0)", "data", ACC_DATASET == "ILSVRC/imagenet-1k", ACC_DATASET)

In [ ]:
# rotary helpers: from W5 / RB-0
def find_rope(model):
    for n, m in model.named_modules():
        if callable(getattr(m, "get_embed", None)): return n, m
    return None, None

def attn_attrs(model, source):
    rh, src, nh = False, "default(interleaved)", None
    for blk in get_vit_blocks(model, source):
        a = get_block_attention(blk, source)
        for at in ("rotate_half", "half", "rope_rotate_half"):
            v = getattr(a, at, None)
            if isinstance(v, bool): rh, src = v, at; break
        nh = getattr(a, "num_heads", None); break
    return rh, src, nh

def plane_columns(p, n, rh): return [p, p + n] if rh else [2 * p, 2 * p + 1]

def derive_axis(emb, n_planes, rh, H, W):
    sin, cos = emb.detach().float().chunk(2, -1); rows = []; dec = True
    for p in range(n_planes):
        c = plane_columns(p, n_planes, rh)[0]
        z = torch.complex(cos[:, c], sin[:, c]).reshape(H, W)
        ra = (z - z.mean(dim=1, keepdim=True)).abs().mean().item()
        rb = (z - z.mean(dim=0, keepdim=True)).abs().mean().item()
        ax = "row" if ra < rb else "col"; lo, hi = sorted([ra, rb]); r = lo / hi if hi > 1e-12 else 1.0
        if r > 0.10: dec = False
        rows.append({"plane": p, "axis": ax, "ratio": r})
    return rows, [r["plane"] for r in rows if r["axis"] == "row"], [r["plane"] for r in rows if r["axis"] == "col"], dec

def mask_planes(emb, kill, n, rh):
    if not kill: return emb
    s, c = emb.chunk(2, -1); s, c = s.clone(), c.clone()
    for p in kill:
        for col in plane_columns(p, n, rh): s[..., col] = 0.0; c[..., col] = 1.0
    return torch.cat([s, c], -1)

def identity_like(emb):
    s, c = emb.chunk(2, -1); return torch.cat([torch.zeros_like(s), torch.ones_like(c)], -1)
IDEN = identity_like

def per_head(heads, inner, n_heads):
    hs = sorted(set(heads))
    def tf(emb):
        if emb.dim() != 2:
            raise RuntimeError(f"per_head expected 2-D rope, got {tuple(emb.shape)}: stacked overrides")
        out = emb.unsqueeze(0).repeat(n_heads, 1, 1)
        if hs:
            t = inner(emb)
            for h in hs: out[h] = t
        return out
    return tf

class RopeComposite:
    def __init__(s, model, source, segs):
        s.blocks = get_vit_blocks(model, source); s.source = source
        s.segs = [(list(b), t) for b, t in segs]; seen = set()
        for b, _ in s.segs:
            if seen & set(b): raise ValueError(f"overlapping segments: {sorted(seen & set(b))}")
            seen |= set(b)
        s.handles = []
    def __enter__(s):
        def mk(tf):
            def pre(mod, args, kw):
                r = kw.get("rope", None)
                if r is not None: kw = dict(kw); kw["rope"] = tf(r)
                return args, kw
            return pre
        for blocks, tf in s.segs:
            for i in blocks:
                s.handles.append(get_block_attention(s.blocks[i], s.source)
                                 .register_forward_pre_hook(mk(tf), with_kwargs=True))
        return s
    def __exit__(s, *e):
        for h in s.handles: h.remove()
        s.handles = []

class NullCtx:
    handles = []
    def __enter__(s): return s
    def __exit__(s, *e): return False

def win_plus(model, source, heads_by_block, n_heads, window_tf=IDEN, inner=IDEN, window=None):
    """Rotations off in the window (window_tf), plus `inner` on the listed heads. heads_by_block: {block: [heads]},
    blocks outside the window."""
    segs = [(list(WINDOW if window is None else window), window_tf)] if (window is None or window) else []
    for k in sorted(heads_by_block):
        hs = sorted(set(heads_by_block[k]))
        if hs: segs.append(([k], per_head(hs, inner, n_heads)))
    return RopeComposite(model, source, segs)

In [ ]:
# algebra gates (as in W5 / RB-0)
def _rot_il(x):
    x = x.unflatten(-1, (-1, 2)); return torch.stack([-x[..., 1], x[..., 0]], -1).flatten(-2)
def _rot_h(x):
    a, b = x.chunk(2, -1); return torch.cat([-b, a], -1)
def _apply(x, e, half=False):
    s, c = e.chunk(2, -1); return x * c + (_rot_h(x) if half else _rot_il(x)) * s

for lname, rh in [("interleaved", False), ("rotate_half", True)]:
    npl, dim, H_ = 32, 64, 12
    e = torch.randn(196, 2 * dim); x = torch.randn(2, H_, 196, dim)
    gate(f"empty plane mask bit-identical [{lname}]", "algebra", torch.equal(mask_planes(e, [], npl, rh), e))
    gate(f"full plane mask equals identity [{lname}]", "algebra",
         torch.allclose(mask_planes(e, list(range(npl)), npl, rh), identity_like(e)))
    m = mask_planes(e, [3], npl, rh)
    touched = torch.nonzero((m - e).abs().sum(0) > 0).flatten().tolist()
    sc = plane_columns(3, npl, rh); exp = sorted(set(sc + [c + dim for c in sc]))
    gate(f"plane 3 touches only its own columns [{lname}]", "algebra", touched == exp, f"{touched} vs {exp}")
    full = _apply(x, e, rh)
    gate(f"mask NO heads equals unmodified rope [{lname}]", "per-head",
         torch.allclose(_apply(x, per_head([], identity_like, H_)(e), rh), full, atol=1e-6))
    one = _apply(x, per_head([3], identity_like, H_)(e), rh); others = [h for h in range(H_) if h != 3]
    gate(f"mask head 3 leaves others untouched [{lname}]", "per-head", torch.allclose(one[:, others], full[:, others], atol=1e-6))
    gate(f"mask head 3 changes head 3 [{lname}]", "per-head", not torch.allclose(one[:, 3], full[:, 3]))
    # single head off after a one-axis window (the window masks that axis's planes, the head masks all)
    ew = mask_planes(e, list(range(0, npl, 2)), npl, rh)
    gate(f"one-axis window leaves the other axis's planes bit-identical [{lname}]", "algebra",
         torch.equal(ew.chunk(2, -1)[0][:, plane_columns(1, npl, rh)], e.chunk(2, -1)[0][:, plane_columns(1, npl, rh)]))

class _Stub:
    def __init__(s, n=12): s.blocks = [nn.Identity() for _ in range(n)]
try:
    _wp = win_plus(_Stub(), "timm", {7: [3]}, 12, window_tf=IDEN)
    dj = [b for b, _ in _wp.segs] == [WINDOW, [7]]
except Exception:
    dj = False
gate("win_plus builds the window plus one segment for the single head's block", "composite", dj)
try:
    win_plus(_Stub(), "timm", {5: [0]}, 12); bad = False
except ValueError:
    bad = True
gate("win_plus REJECTS a single head inside the window", "composite", bad)

In [ ]:
# image cache: decode once per model, replay through a real DataLoader (RB-0)
class _CachedBatches(torch.utils.data.IterableDataset):
    def __init__(self, batches): super().__init__(); self.batches = batches
    def __iter__(self):
        for b in self.batches: yield b

def cached_loader(cache):
    return torch.utils.data.DataLoader(_CachedBatches(cache), batch_size=None, num_workers=0,
                                       pin_memory=torch.cuda.is_available(), shuffle=False)

def build_cache(dataset, processor, source, n):
    t = time.time()
    dl = prep_data(dataset, processor, source, corruption_type=None, number_images=n,
                   batch_size=BATCH_SIZE, half=False, num_workers=0)
    out = [(im.clone(), torch.as_tensor(lb).clone()) for im, lb in dl]
    got = sum(int(lb.numel()) for _, lb in out)
    print(f"    cached {got} images in {time.time() - t:.0f}s")
    return out

def cache_digest(cache):
    h = hashlib.sha1()
    for im, lb in cache[:2]:
        h.update(im[:4].numpy().tobytes()); h.update(lb[:4].numpy().tobytes())
    return h.hexdigest()[:12]

In [ ]:
# measurement: RB-0's measure() without the bootstrap / head-norm extras
def final_norm_module(model):
    n = getattr(model, "norm", None)
    if n is not None and not isinstance(n, nn.Identity): return n, "norm"
    f = getattr(model, "fc_norm", None)
    if f is not None and not isinstance(f, nn.Identity): return f, "fc_norm"
    return nn.Identity(), "identity"

def measure(M, ctx, capture=()):
    """One RPI sweep over the cached images under `ctx`: SSDC at every block input and at FINAL, plus captured patch
    tokens."""
    model, source, n_prefix = M["model"], M["source"], M["n_prefix"]
    cache = M["cache"]["ssdc"]
    blocks = get_vit_blocks(model, source)
    acc = SSDCAccumulator(n_prefix=n_prefix, spatial_metric=METRIC)
    handles = _register_ssdc_accumulator_hooks(model, source, acc)        # the repo's own readout
    fnorm = M["final_norm"]
    st = {"next": 0, "cur": 0}
    cap = {r: [] for r in capture}
    def keep(r, tok):
        if r in cap: cap[r].append(tok[:, n_prefix:, :].detach().float().cpu())
    extra = [model.register_forward_pre_hook(
        lambda m, args: st.update(cur=st["next"], next=st["next"] + int(args[0].shape[0])))]
    for i in range(N_BLOCKS):
        if i in cap:
            extra.append(get_block_attention(blocks[i], source).register_forward_hook(
                lambda m, inp, out, i=i: keep(i, inp[0].detach())))
    def final_hook(m, inp, out):
        t = fnorm(out.detach()); acc.add(FINAL, t); keep(FINAL, t)
    extra.append(blocks[-1].register_forward_hook(final_hook))
    t0 = time.time()
    torch.manual_seed(TORCH_SEED); np.random.seed(TORCH_SEED % (2 ** 31))
    try:
        with ctx:
            hooks = len(getattr(ctx, "handles", []) or [])
            predict(model, cached_loader(cache), source, RPI=True, half=False)
    finally:
        for h in handles + extra: h.remove()
    ssdc = [acc.ssdc(i, remove_prefix=True) for i in range(N_BLOCKS)] + [acc.ssdc(FINAL, remove_prefix=True)]
    res = {"ssdc": [float(v) for v in ssdc], "hooks": hooks, "n_images": int(st["next"]),
           "runtime_s": round(time.time() - t0, 1)}
    if capture:
        res["acts"] = {r: torch.cat(cap[r], 0) for r in capture}
    return res

# probes: W5's per-block-seeded protocol from RB-0
def _fit(Xtr, ytr, Xte, yte, nc, seed):
    torch.manual_seed(seed)
    head = nn.Linear(Xtr.shape[1], nc).to(Xtr.device)
    opt = torch.optim.AdamW(head.parameters(), lr=PROBE_LR)
    n = Xtr.shape[0]; peak = 0.0; best = None
    for _ in range(PROBE_PASSES):
        perm = torch.randperm(n, device=Xtr.device); head.train()
        for s0 in range(0, n, PROBE_BATCH):
            b = perm[s0:s0 + PROBE_BATCH]
            opt.zero_grad(); F.cross_entropy(head(Xtr[b]), ytr[b]).backward(); opt.step()
        head.eval(); correct = 0; preds = []
        with torch.no_grad():
            for s0 in range(0, Xte.shape[0], PROBE_CHUNK):
                p = head(Xte[s0:s0 + PROBE_CHUNK]).argmax(-1); preds.append(p)
                correct += (p == yte[s0:s0 + PROBE_CHUNK]).sum().item()
        acc_ = correct / Xte.shape[0]
        if acc_ > peak: peak = acc_; best = torch.cat(preds)
    del head, opt
    return float(peak), best

def _split(n_images, T, seed=TORCH_SEED):
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(n_images, generator=g); n_tr = int(round(n_images * PROBE_TRAIN_FRAC))
    rows = lambda ids: (ids.unsqueeze(1) * T + torch.arange(T).unsqueeze(0)).reshape(-1)
    return rows(idx[:n_tr]), rows(idx[n_tr:]), idx[n_tr:]

def probe_block(acts, H, W, block, heads=("exact", "row", "col"), noise=False):
    n_images, T, D = acts.shape
    feats = acts.reshape(-1, D)
    tr, te, te_img = _split(n_images, T)
    Xtr = feats[tr].to(PDEV); Xte = feats[te].to(PDEV)
    mu = Xtr.mean(0, keepdim=True); sd = Xtr.std(0, keepdim=True).clamp_min(1e-6)
    Xtr = (Xtr - mu) / sd; Xte = (Xte - mu) / sd
    slot = torch.arange(T); out = {}
    labels = {"exact": (slot, T), "row": (slot // W, H), "col": (slot % W, W)}
    for name in heads:
        lab, nc = labels[name]; y = lab.repeat(n_images)
        acc_, preds = _fit(Xtr, y[tr].to(PDEV), Xte, y[te].to(PDEV), nc, TORCH_SEED + block)
        out[name] = acc_
        if noise:
            corr = (preds == y[te].to(PDEV)).float()
            g = np.random.default_rng(TORCH_SEED + block)
            n_te = len(te_img); per = corr.reshape(n_te, T).mean(1).cpu().numpy()
            bt = [per[g.integers(0, n_te, n_te)].mean() for _ in range(NOISE_BOOT)]
            seeds = [_fit(Xtr, y[tr].to(PDEV), Xte, y[te].to(PDEV), nc, TORCH_SEED + block + 1000 * s)[0]
                     for s in NOISE_SEEDS]
            out.setdefault("noise", {})[name] = {
                "boot_sd": float(np.std(bt, ddof=1)), "seed_sd": float(np.std(seeds, ddof=1)),
                "seed_values": [float(v) for v in seeds],
                "combined_sd": float(np.sqrt(np.var(bt, ddof=1) + np.var(seeds, ddof=1)))}
    out["chance"] = {"exact": 1.0 / T, "row": 1.0 / H, "col": 1.0 / W}
    del Xtr, Xte
    if PDEV == "cuda": torch.cuda.empty_cache()
    return out

In [ ]:
# top-1 on the cached ILSVRC images: RB-0's accuracy()
def accuracy(M, ctx):
    model, cache = M["model"], M["cache"]["acc"]
    model.eval(); correct = []
    torch.manual_seed(TORCH_SEED)
    with ctx:
        hooks = len(getattr(ctx, "handles", []) or [])
        with torch.inference_mode():
            for images, labels in cached_loader(cache):
                pred = model(images.to(DEVICE)).argmax(-1).cpu()
                correct.append(pred == torch.as_tensor(labels).cpu())
    c = torch.cat(correct)
    return float(c.float().mean()), c.numpy().astype(np.uint8), hooks

In [ ]:
# per-model setup (RB-2's, plus the ILSVRC cache and RB-0's top-1 replay gate)
TEST_INIT_SEED = {"naver": 101, "DINOv3": 102, "small": 103, "reg1_gap": 104}
def load_model(key):
    spec = MODEL_SPECS[key]
    if TEST_MODE:
        tiny = dict(embed_dim=96, num_heads=6 if key != "naver" else 12, pretrained=False, dynamic_img_size=True)
        if key == "naver": tiny["embed_dim"] = 192
        tiny["num_classes"] = 10 if spec["classifier"] else 0
        torch.manual_seed(TEST_INIT_SEED[key])
        model = timm.create_model(spec["name"].split(".")[0], **tiny).to(DEVICE).eval()
        with torch.no_grad():
            for blk in model.blocks:
                for gname in ("gamma_1", "gamma_2"):
                    g_ = getattr(blk, gname, None)
                    if isinstance(g_, torch.Tensor): g_.fill_(1.0)
                q_ = getattr(blk.attn, "qkv", None)
                if q_ is not None: q_.weight.mul_(4.0)
                for pn in ("q_proj", "k_proj"):
                    m_ = getattr(blk.attn, pn, None)
                    if m_ is not None: m_.weight.mul_(4.0)
        cfg = timm.data.resolve_model_data_config(model); cfg["input_size"] = (3, INPUT_SIZE, INPUT_SIZE)
        return model, timm.data.create_transform(**cfg, is_training=False), "timm"
    return load_rope(model_name=spec["name"], device=DEVICE, half=False, input_size=INPUT_SIZE)

def first_layer_rel_diff(M):
    model, source = M["model"], M["source"]
    conv = get_patch_embed_conv(model, source); grab = {}
    h = conv.register_forward_hook(lambda m, i, o: grab.update(x=i[0].detach(), y=o.detach()))
    try:
        with torch.no_grad(): model(M["cache"]["ssdc"][0][0].to(DEVICE))
    finally:
        h.remove()
    with torch.no_grad():
        y64 = F.conv2d(grab["x"].double(), conv.weight.double(), None if conv.bias is None else conv.bias.double(),
                       conv.stride, conv.padding, conv.dilation, conv.groups)
        v = float((grab["y"].double() - y64).abs().max() / y64.abs().max())
    n = int(grab["x"].shape[0]); del grab, y64
    return v, n

def setup_model(key):
    spec = MODEL_SPECS[key]
    print("\n" + "=" * 78 + f"\n{key}  ({spec['name']})\n" + "=" * 78, flush=True)
    model, processor, source = load_model(key)
    model.eval().float()
    rh, rh_src, nh_attr = attn_attrs(model, source)
    n_prefix = int(getattr(model, "num_prefix_tokens", 1))
    M = {"key": key, "model": model, "source": source, "processor": processor, "n_prefix": n_prefix,
         "rotate_half": bool(rh), "n_heads": int(nh_attr), "cache": {}}
    M["cache"]["ssdc"] = build_cache(DS_SSDC, processor, source, N_SSDC)
    if spec["classifier"]:
        M["cache"]["acc"] = build_cache(DS_ACC, processor, source, N_ACC)
    with torch.no_grad():
        g = get_patch_embed_conv(model, source)(M["cache"]["ssdc"][0][0][:1].to(DEVICE))
    Hg, Wg = int(g.shape[-2]), int(g.shape[-1]); M["grid"] = (Hg, Wg); T = Hg * Wg
    gate("forward grid is 14x14", key, (Hg, Wg) == (14, 14), f"{Hg}x{Wg}")
    fn, fn_name = final_norm_module(model); M["final_norm"] = fn; M["final_norm_name"] = fn_name
    A_ = ANCH.get(key)
    if A_:
        gate("num_heads matches W5", key, M["n_heads"] == A_["n_heads"], f"{M['n_heads']} vs {A_['n_heads']}")
    ra = (RB0.get(key) or {}).get("A")
    if ra:
        gate("num_heads matches RB-0", key, M["n_heads"] == ra["n_heads"], f"{M['n_heads']} vs {ra['n_heads']}")
    rname, rmod = find_rope(model); M["rope"] = rmod
    emb0 = rmod.get_embed(shape=(Hg, Wg))
    n_planes = int(emb0.shape[-1] // 4)
    amap, rowp, colp, dec = derive_axis(emb0, n_planes, rh, Hg, Wg)
    gate("rope rows equal patch tokens (196)", key, int(emb0.shape[0]) == T == 196, f"{emb0.shape[0]} vs {T}")
    gate("rotary table has no head dimension", key, emb0.dim() == 2, f"shape {tuple(emb0.shape)}")
    if A_:
        gate("row/col planes equal W5's runtime derivation", key, rowp == A_["row_planes"] and colp == A_["col_planes"], f"row {rowp[:3]}..")
    if DEVICE == "cuda" and not TEST_MODE:
        v, nb = first_layer_rel_diff(M); exp_ = FIRST_LAYER_910[key]; M["first_layer_rel_diff"] = v
        gate("first-layer numerics equal RB-1b's cuDNN 9.10.2 value (TF32-level kernel, as in September)", key,
             nb == BATCH_SIZE and abs(v / exp_ - 1) <= 0.01, f"{v:.4e} vs {exp_:.4e} (bit-identical: {v == exp_}; batch {nb})")
    else:
        gate("first-layer numerics equal RB-1b's cuDNN 9.10.2 value", key, True, "test mode / CPU", skip=True)
    def streamed_ssdc(ctx):
        torch.manual_seed(TORCH_SEED); np.random.seed(TORCH_SEED % (2 ** 31))
        with ctx:
            s, _ = evaluate_ssdc(model, processor, DS_SSDC, source, RPI=True, number_images=N_SSDC,
                                 batch_size=BATCH_SIZE, metric=METRIC, half=False, num_workers=0)
        return s
    for nm_, mk in [("baseline", NullCtx), ("Cref", lambda: RopeComposite(model, source, [(WINDOW, IDEN)]))]:
        a = streamed_ssdc(mk()); b = measure(M, mk())["ssdc"][:N_BLOCKS]
        d = max(abs(x - y) for x, y in zip(a, b))
        gate(f"cached replay reproduces streamed evaluate_ssdc [{nm_}]", key, d <= EXACT_TOL, f"max |delta| {d:.2e}")
    if "acc" in M["cache"]:
        n_chk = min(REPLAY_CHECK_N, N_ACC)
        dl = prep_data(DS_ACC, processor, source, corruption_type=None, number_images=n_chk,
                       batch_size=BATCH_SIZE, half=False, num_workers=0)
        cs = []
        with torch.inference_mode():
            for im, lb in dl:
                cs.append(model(im.to(DEVICE)).argmax(-1).cpu() == torch.as_tensor(lb))
        cs = torch.cat(cs).numpy().astype(np.uint8)
        _, cc, _ = accuracy(M, NullCtx())
        gate(f"cached replay reproduces streamed top-1 on the first {n_chk} images", key,
             np.array_equal(cs, cc[:n_chk]), f"{int((cs != cc[:n_chk]).sum())} images differ")
    return M

In [ ]:
# bookkeeping and the fixed plan of arms
RESULTS = {}
CFG_FP = {"n_ssdc": N_SSDC, "n_acc": N_ACC, "batch": BATCH_SIZE, "seed": TORCH_SEED, "ssdc_dataset": SSDC_DATASET,
          "acc_dataset": ACC_DATASET, "metric": METRIC, "windows": WINDOWS, "test_mode": TEST_MODE}
SESSION = {"session": RUN_STAMP, "torch": torch.__version__, "cudnn": ENV["cudnn_reported"], "python": sys.executable,
           "gpu": ENV["gpu"], "repo_commit": REPO_COMMIT}
def model_file(key): return OUT_DIR / f"rb5_{key}.json"
def img_file(key): return OUT_DIR / f"rb5_perimage_{key}.npz"

def load_R(key):
    f = model_file(key)
    if not (RESUME and f.exists()): return None, {}
    R = json.loads(f.read_text())
    if R.get("config") != CFG_FP:
        raise SystemExit(f"{f.name} was written with a different configuration; move it away or set RB5_RESUME=0.")
    P = {n: z_[n] for z_ in [np.load(img_file(key))] for n in z_.files} if img_file(key).exists() else {}
    for n in R["acc"]:
        if n not in P: raise SystemExit(f"{img_file(key).name} lacks per-image results for {n}; set RB5_RESUME=0.")
    return R, P

def save_R(key, R, P):
    R["saved_utc"] = datetime.now(timezone.utc).isoformat()
    if P:
        tmpi = img_file(key).with_name(img_file(key).stem + ".tmp.npz"); np.savez_compressed(tmpi, **P); os.replace(tmpi, img_file(key))
    tmp = model_file(key).with_suffix(".json.tmp"); tmp.write_text(json.dumps(R, indent=1, default=float)); os.replace(tmp, model_file(key))

def first_after(w): return WINDOWS[w][-1] + 1
def singles_of(w, nh):
    blocks = [6] if w == "w5" else list(range(first_after(w), N_BLOCKS))     # w5: block 6 only, as a gate on RB-0
    return [(k, h) for k in blocks for h in range(nh)]

def plan(key, nh):
    arms = {"ssdc_baseline": ("ssdc", ("null",))}
    for w in WINDOWS:
        arms[f"ssdc_ref_{w}"] = ("ssdc", ("window", w))
        for k, h in singles_of(w, nh): arms[f"ssdc_{w}_b{k}_h{h}"] = ("ssdc", ("single", w, k, h))
    if MODEL_SPECS[key]["classifier"]:
        arms["baseline"] = ("acc", ("null",))
        for w in WINDOWS:
            arms[f"ref_{w}"] = ("acc", ("window", w))
            for k, h in singles_of(w, nh): arms[f"{w}_b{k}_h{h}"] = ("acc", ("single", w, k, h))
    return arms

def build_ctx(M, spec):
    model, source, nh = M["model"], M["source"], M["n_heads"]
    if spec[0] == "null": return NullCtx()
    if spec[0] == "window": return RopeComposite(model, source, [(WINDOWS[spec[1]], IDEN)])
    if spec[0] == "single":
        _, w, k, h = spec; return win_plus(model, source, {k: [h]}, nh, window=WINDOWS[w])
    raise ValueError(spec)
def expected_hooks(spec):
    return {"null": 0, "window": len(WINDOWS[spec[1]]) if spec[0] == "window" else 0,
            "single": (len(WINDOWS[spec[1]]) + 1) if spec[0] == "single" else 0}[spec[0]]

## Gates on the saved results
All read the saved file, so a resumed or re-loaded run still checks every arm. "Bit for bit" = identical values.

In [ ]:
def _maxdiff(a, b): return max((abs(x - y) for x, y in zip(a, b)), default=0.0)

def check_refs(key, R, P):
    ok = []; rb = RB0[key]; ca = rb["A"]["conditions"]; s = R["ssdc"]
    d = max(_maxdiff(s["ssdc_baseline"]["ssdc"], ca["baseline"]["ssdc"]), _maxdiff(s["ssdc_ref_w5"]["ssdc"], ca["Cref"]["ssdc"]))
    ok.append(gate("baseline and 0-5 window SSDC equal RB-0 part A bit for bit (13 readouts)", key, d <= EXACT_TOL, f"max |delta| {d:.1e}"))
    if key in W3:
        d = max(_maxdiff(s[f"ssdc_ref_{w}"]["ssdc"][:N_BLOCKS], W3[key][w]) for w in WINDOWS)
        ok.append(gate("0-3, 0-5 and 0-7 window SSDC equal W3 prefix_3 / prefix_5 / prefix_7 bit for bit (12 readouts)", key,
                       d <= EXACT_TOL, ", ".join(f"{w} b11 {s[f'ssdc_ref_{w}']['ssdc'][11]:.6f}" for w in WINDOWS) + f"; max |delta| {d:.1e}"))
    elif not TEST_MODE:
        gate("0-3 / 0-7 window references: no W3 anchor for this model (W3 ran naver and DINOv3 only); same code path as the W3-gated models", key, True)
    if MODEL_SPECS[key]["classifier"]:
        cc, Pc = rb["C"]["conditions"], rb["P"]
        bad = [a for a, b in (("baseline", "baseline"), ("ref_w5", "window")) if R["acc"][a]["acc"] != cc[b]["acc"] or not np.array_equal(P[a], Pc[b])]
        ok.append(gate("top-1 baseline and 0-5 window equal RB-0 part C bit for bit (mean and per image)", key, not bad,
                       ", ".join(f"{n} {R['acc'][n]['acc'] * 100:.2f}" for n in ("baseline", "ref_w3", "ref_w5", "ref_w7") if n in R["acc"])))
    return all(ok)

def check_arms(key, R, P, nh):
    arms = plan(key, nh); rb = RB0[key]
    miss = [n for n, (kind, _) in arms.items() if n not in R[kind]]
    gate("every planned arm measured", key, not miss, f"{len(arms) - len(miss)}/{len(arms)}" + (f"; missing {miss[:4]}" if miss else ""))
    badh = [n for n, (kind, spec) in arms.items() if n in R[kind] and R[kind][n]["hooks"] != expected_hooks(spec)]
    gate("every arm registered exactly the expected hooks", key, not badh, ",".join(badh[:5]))
    badn = [n for n, (kind, _) in arms.items() if n in R[kind] and R[kind][n]["n_images"] != (N_ACC if kind == "acc" else N_SSDC)]
    gate("every arm saw every image", key, not badn, ",".join(badn[:5]))
    s = R["ssdc"]; ca = rb["A"]["conditions"]
    d = max(_maxdiff(s[f"ssdc_w5_b6_h{h}"]["ssdc"], ca[f"s_b6_h{h}"]["ssdc"]) for h in range(nh) if f"ssdc_w5_b6_h{h}" in s)
    gate("block-6 single heads under the 0-5 window equal RB-0 part A bit for bit (SSDC, 13 readouts)", key, d <= EXACT_TOL, f"max |delta| {d:.1e}")
    worst, inert = 0.0, []
    for n, (kind, spec) in arms.items():
        if kind == "ssdc" and spec[0] == "single" and n in s:
            _, w, k, h = spec; ref = s[f"ssdc_ref_{w}"]["ssdc"]
            worst = max(worst, _maxdiff(s[n]["ssdc"][:k + 1], ref[:k + 1]))
            if s[n]["ssdc"][FINAL] == ref[FINAL]: inert.append(n)
    gate("single-head masks leave every readout at or before their block bit-identical to their window's reference", key, worst <= EXACT_TOL, f"max |delta| {worst:.1e}")
    gate("every single-head mask changes the SSDC at the output of block 11 (no inert mask)", key, not inert, ",".join(inert[:5]))
    if MODEL_SPECS[key]["classifier"]:
        cc, Pc = rb["C"]["conditions"], rb["P"]
        bad = [h for h in range(nh) if f"w5_b6_h{h}" in R["acc"] and (R["acc"][f"w5_b6_h{h}"]["acc"] != cc[f"s_b6_h{h}"]["acc"]
                                                                    or not np.array_equal(P[f"w5_b6_h{h}"], Pc[f"s_b6_h{h}"]))]
        gate("block-6 single heads under the 0-5 window equal RB-0 part C bit for bit (top-1, mean and per image)", key, not bad, str(bad))
    if not TEST_MODE:
        ses = R.get("sessions", [])
        gate("every session that wrote this file ran torch 2.10.0+cu128 with cuDNN 91002", key,
             bool(ses) and all(x.get("cudnn") == EXPECT_CUDNN and x.get("torch") == EXPECT_TORCH for x in ses), f"{len(ses)} session(s)")

## Measurement
Per model: references (baseline, the three windows) and their gates first (a failure stops that model when
`RB5_STRICT=1`), then every planned arm. Saves every 24 arms; a re-run continues where it stopped.

In [ ]:
def run_model(M):
    key, nh = M["key"], M["n_heads"]
    R, P = load_R(key)
    if R is None:
        R, P = {"model": key, "name": MODEL_SPECS[key]["name"], "notebook": "RB-5", "config": CFG_FP, "n_heads": nh,
                "acc": {}, "ssdc": {}, "sessions": []}, {}
    else:
        print(f"  resuming {model_file(key).name}: {len(R['acc'])} top-1 arms, {len(R['ssdc'])} SSDC arms saved")
    R["sessions"].append({**SESSION, "first_layer_rel_diff": M.get("first_layer_rel_diff")})
    arms = plan(key, nh); t0 = time.time(); n_new = [0]
    def run(name):
        kind, spec = arms[name]
        if name in R[kind]: return
        ctx = build_ctx(M, spec)
        if kind == "acc":
            a, c, hooks = accuracy(M, ctx); R["acc"][name] = {"acc": a, "hooks": hooks, "n_images": int(len(c)), "spec": list(spec)}; P[name] = c
        else:
            r = measure(M, ctx); R["ssdc"][name] = {"ssdc": r["ssdc"], "hooks": r["hooks"], "n_images": r["n_images"], "spec": list(spec)}
        n_new[0] += 1
    refs = [n for n in arms if n in ("ssdc_baseline", "baseline") or "_ref_" in n or n.startswith("ref_")]
    for n in refs: run(n)
    save_R(key, R, P)
    if not check_refs(key, R, P) and STRICT and not TEST_MODE:
        gate("arms measured", key, False, "STOPPED: a reference gate failed (RB5_STRICT=1). Send the gate report.")
        return R
    todo = [n for n in arms if n not in refs and n not in R[arms[n][0]]]
    print(f"\n--- {key}: {len(todo)} arms to measure ---", flush=True)
    for i, n in enumerate(todo):
        run(n)
        if (i + 1) % 24 == 0 or i == len(todo) - 1:
            save_R(key, R, P); el = time.time() - t0
            print(f"    {i + 1}/{len(todo)} arms | {el / 60:.1f} min, ~{el / (i + 1) * (len(todo) - i - 1) / 60:.0f} min left", flush=True)
    R["complete"] = all(n in R[arms[n][0]] for n in arms)
    save_R(key, R, P)
    check_arms(key, R, P, nh)
    gate("model finished", key, True, f"{n_new[0]} new arms, {(time.time() - t0) / 60:.1f} min; complete {R['complete']}")
    return R

In [ ]:
T_RUN = time.time()
for key in MODELS_RUN:
    if key not in SEL:
        gate("model measured", key, False, "no RB-0 part A selection"); continue
    R0, P0 = load_R(key)
    if R0 and R0.get("complete"):
        print(f"\n{key}: already complete in {model_file(key).name}; loading and re-checking")
        RESULTS[key] = R0; check_refs(key, R0, P0); check_arms(key, R0, P0, R0["n_heads"]); continue
    try:
        M = setup_model(key)
    except Exception as ex:
        traceback.print_exc(); gate("model setup completed", key, False, repr(ex)[:200]); continue
    setup_fail = [g for g in GATES if g["scope"] == key and g["status"] == "FAIL"]
    if setup_fail and STRICT and not TEST_MODE:
        gate("model measured", key, False, f"STOPPED: {len(setup_fail)} setup gate(s) failed: " + "; ".join(g["gate"] for g in setup_fail)[:300])
        del M; continue
    try:
        RESULTS[key] = run_model(M)
    except Exception as ex:
        traceback.print_exc(); gate("model measured", key, False, repr(ex)[:200])
    del M
    if DEVICE == "cuda": torch.cuda.empty_cache()
print(f"\nmeasurement finished in {(time.time() - T_RUN) / 60:.1f} min")

## Analysis
Reads only saved files. For the 0-5 window every single head comes from RB-0 (block 6 re-measured and gated above).

In [ ]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

def paired(a, b):
    d = a.astype(float) - b.astype(float); return float(d.mean() * 100), float(d.std(ddof=1) / np.sqrt(len(d)) * 100)
def perm_spearman(x, y, n_perm=PERM_N, seed=0):
    x, y = np.asarray(x, float), np.asarray(y, float)
    rx, ry = rankdata(x), rankdata(y)
    if len(x) < 3 or np.std(rx) == 0 or np.std(ry) == 0: return float("nan"), float("nan")
    rho = float(np.corrcoef(rx, ry)[0, 1]); g = np.random.default_rng(seed)
    null = np.array([np.corrcoef(rx, g.permutation(ry))[0, 1] for _ in range(n_perm)])
    return rho, float((np.sum(np.abs(null) >= abs(rho)) + 1) / (n_perm + 1))
def show(title, obj):
    print("\n" + title); print("-" * len(title))
    if isinstance(obj, pd.DataFrame): print(obj.to_string(index=False) if len(obj) else "(empty)")
    else: print(json.dumps(obj, indent=1, default=float))

ANALYSIS, TABLES = {}, {"singles": [], "blocks": []}
for key in MODELS_RUN:
    R = RESULTS.get(key) or (json.loads(model_file(key).read_text()) if model_file(key).exists() else None)
    if not R or "ssdc_ref_w3" not in R["ssdc"]: print(f"{key}: nothing usable saved"); continue
    P = {n: z_[n] for z_ in [np.load(img_file(key))] for n in z_.files} if img_file(key).exists() else {}
    rb = RB0[key]; ca = rb["A"]["conditions"]; nh = R["n_heads"]; s = R["ssdc"]; cls_ = MODEL_SPECS[key]["classifier"]
    rows = []
    for w in WINDOWS:
        if w == "w5":     # 0-5: every single head from RB-0 (gated above on block 6)
            ref = ca["Cref"]["ssdc"]; heads = [(k, h) for k in POST for h in range(nh)]
            get_s = lambda k, h: ca[f"s_b{k}_h{h}"]["ssdc"]
            Pc = rb.get("P"); refacc = Pc["window"] if (cls_ and Pc is not None) else None
            get_a = (lambda k, h: Pc[f"s_b{k}_h{h}"]) if refacc is not None else None
        else:
            ref = s[f"ssdc_ref_{w}"]["ssdc"]; heads = singles_of(w, nh)
            get_s = lambda k, h, w=w: s[f"ssdc_{w}_b{k}_h{h}"]["ssdc"]
            refacc = P.get(f"ref_{w}"); get_a = (lambda k, h, w=w: P[f"{w}_b{k}_h{h}"]) if refacc is not None else None
        for k, h in heads:
            ss = get_s(k, h)
            r_ = {"model": key, "window": w, "block": k, "head": h, "first_block_after_window": first_after(w),
                  "blocks_after_edge": k - first_after(w), "ssdc_cost_b11": ref[11] - ss[11] if k < 11 else float("nan"),
                  "ssdc_cost_out11": ref[FINAL] - ss[FINAL]}
            if get_a is not None:
                r_["top1_cost"], r_["top1_se"] = paired(refacc, get_a(k, h))
            rows.append(r_)
    df = pd.DataFrame(rows); TABLES["singles"] += rows
    out = {}
    top05 = EXPECT_TOP[key][0] if not TEST_MODE else tuple(SEL[key]["top"][0])
    for w in WINDOWS:
        d = df[df.window == w]; d610 = d[d.block < 11].sort_values("ssdc_cost_b11", ascending=False).reset_index(drop=True)
        o = {"first_block_after_window": first_after(w), "top5_by_ssdc": [[int(a), int(b), round(float(c), 5)] for a, b, c in d610[["block", "head", "ssdc_cost_b11"]].values[:5]],
             "top_head_in_first_block": bool(len(d610) and int(d610.block.iloc[0]) == first_after(w))}
        blk = d[d.block < 11].groupby("block").ssdc_cost_b11.apply(lambda x: x.clip(lower=0).sum())
        o["share_of_positive_ssdc_cost_by_block"] = {int(b): round(float(v / blk.sum()), 3) for b, v in blk.items()} if blk.sum() > 0 else {}
        for b, v in blk.items():
            TABLES["blocks"].append({"model": key, "window": w, "block": int(b), "blocks_after_edge": int(b) - first_after(w),
                                     "positive_ssdc_cost_b11": float(v), "share": float(v / blk.sum()) if blk.sum() > 0 else float("nan")})
        if top05[0] >= first_after(w) and top05[0] < 11:
            hit = d610.index[(d610.block == top05[0]) & (d610["head"] == top05[1])]
            o["rank_of_0to5_top_head_by_ssdc"] = [int(hit[0]) + 1, int(len(d610))] if len(hit) else None
        if "top1_cost" in d:
            dt = d.sort_values("top1_cost", ascending=False).reset_index(drop=True)
            o["top5_by_top1"] = [[int(a), int(b), round(float(c), 2), round(float(e), 2)] for a, b, c, e in dt[["block", "head", "top1_cost", "top1_se"]].values[:5]]
            o["costliest_top1_in_first_block"] = bool(int(dt.block.iloc[0]) == first_after(w))
            if top05[0] >= first_after(w):
                hit = dt.index[(dt.block == top05[0]) & (dt["head"] == top05[1])]
                o["rank_of_0to5_top_head_by_top1"] = [int(hit[0]) + 1, int(len(dt))] if len(hit) else None
                o["0to5_top_head_top1_cost"] = [float(dt.loc[hit[0], "top1_cost"]), float(dt.loc[hit[0], "top1_se"])] if len(hit) else None
            o["window_top1"] = (float(P[f"ref_{w}"].mean() * 100) if w != "w5" else float(rb["P"]["window"].mean() * 100))
        o["window_ssdc_b11"] = float((s[f"ssdc_ref_{w}"]["ssdc"] if w != "w5" else ca["Cref"]["ssdc"])[11])
        out[w] = o
    # single-head costs across windows, on heads both windows measure (blocks <= 10 for SSDC)
    cons = {}
    for a, b in (("w3", "w5"), ("w5", "w7"), ("w3", "w7")):
        x = df[df.window == a].set_index(["block", "head"]); y = df[df.window == b].set_index(["block", "head"])
        common = [i for i in x.index if i in y.index and i[0] < 11]
        if common:
            cons[f"{a}_vs_{b}"] = {"blocks": sorted({int(i[0]) for i in common}), "n": len(common),
                                   "ssdc_b11": perm_spearman(x.loc[common, "ssdc_cost_b11"], y.loc[common, "ssdc_cost_b11"])}
            if "top1_cost" in x and "top1_cost" in y:
                cons[f"{a}_vs_{b}"]["top1"] = perm_spearman(x.loc[common, "top1_cost"], y.loc[common, "top1_cost"])
    out["consistency_across_windows"] = cons
    w3 = out["w3"]; r_s = w3.get("rank_of_0to5_top_head_by_ssdc"); r_a = w3.get("rank_of_0to5_top_head_by_top1")
    fixed = bool(r_s and r_s[0] <= 3 and (not cls_ or (r_a and r_a[0] <= 3)))
    edge = bool(w3["top_head_in_first_block"] and r_s and r_s[0] > 10)
    out["reading"] = {"rule": RULE_READING, "0to5_top_head": list(top05), "rank_under_0to3_ssdc": r_s, "rank_under_0to3_top1": r_a,
                      "verdict": "FIXED HEAD" if fixed else ("EDGE-FOLLOWING" if edge else "MIXED")}
    ANALYSIS[key] = out

for key, out in ANALYSIS.items():
    print("\n" + "=" * 78 + f"\n{key}  ({MODEL_SPECS[key]['label']})   reading: {out['reading']['verdict']}\n" + "=" * 78)
    show("pre-registered reading", {k: v for k, v in out["reading"].items() if k != "rule"})
    for w in WINDOWS: show(f"window {WINDOWS[w][0]}-{WINDOWS[w][-1]} (first block after it: {first_after(w)})", out[w])
    show("do single-head costs agree across windows? (Spearman, permutation p, shared heads)", out["consistency_across_windows"])

## Gate report and export

In [ ]:
_all = []
for f in sorted(OUT_DIR.glob("rb5_gates_session_*.json")): _all += json.loads(f.read_text())
_seen = {}
for g in _all: _seen[(g["scope"], g["gate"])] = g
gdf = pd.DataFrame(list(_seen.values()))
print(f"GATE REPORT  ({len(set(g['session'] for g in _all))} session(s), {len(_all)} gate calls, latest result per gate shown)\n")
print(gdf.to_string(index=False) if len(gdf) else "(none)")
n_fail = int((gdf.status == "FAIL").sum()) if len(gdf) else 0
n_skip = int((gdf.status == "SKIP").sum()) if len(gdf) else 0
complete = {k: bool((RESULTS.get(k) or {}).get("complete")) or (model_file(k).exists() and json.loads(model_file(k).read_text()).get("complete", False)) for k in MODELS_RUN}
print(f"\n{len(gdf)} gates: {int((gdf.status == 'PASS').sum())} pass, {n_fail} fail, {n_skip} skipped | complete {complete}")
print("=" * 78)
print("ALL GATES PASSED" if n_fail == 0 else f"{n_fail} GATE(S) FAILED: report them before anyone uses the numbers")
for _, r in gdf[gdf.status == "FAIL"].iterrows(): print(f"   - [{r['scope']}] {r['gate']}: {r['detail']}")
print("=" * 78)
export = {"notebook": "RB-5", "generated_utc": RUN_STAMP, "test_mode": TEST_MODE, "environment": ENV, "repo": str(repo), "repo_commit": REPO_COMMIT,
          "rb0_dir": str(_rb0) if _rb0 else None, "config": {**CFG_FP, "models": MODELS_RUN, "strict": STRICT},
          "rules": {"rank": RULE_RANK, "reading": RULE_READING}, "analysis": ANALYSIS, "gates": gdf.to_dict("records"),
          "gate_summary": {"total": len(gdf), "fail": n_fail, "skip": n_skip, "complete": complete, "sessions": sorted(set(g["session"] for g in _all))}}
(OUT_DIR / f"rb5_all_{RUN_STAMP}.json").write_text(json.dumps(export, indent=1, default=float))
gdf.to_csv(OUT_DIR / "rb5_gate_report.csv", index=False)
for name, rows in TABLES.items():
    if rows: pd.DataFrame(rows).to_csv(OUT_DIR / f"rb5_{name}.csv", index=False)
archive = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
print("written:")
for p in sorted(OUT_DIR.iterdir()): print(f"  {p.name}  ({p.stat().st_size:,} bytes)")
print(f"\narchive: {archive} ({os.path.getsize(archive):,} bytes)")